# 🏗️ Data Foundation | Notebook 03
## Modelagem Dimensional em PySpark — Star Schema

---

**Série:** #DataFoundation  
**Autor:** Léo Touguinha | Especialista em Dados | Mercado Financeiro  
**Pré-requisito:** Notebook 01 executado

---

## Contexto

> Este modelo foi desenhado com base em operação real: funil de originação com 100.000 cotações/mês e 3 estágios de conversão. Os dados são sintéticos. Os parâmetros de negócio são reais.
>
> **Nota:** Os notebooks 01 e 02 simulam uma carteira de crédito/transações financeiras. Os notebooks 03 e 04 simulam uma corretora de seguro garantia. São dois estudos de caso distintos dentro do mesmo pipeline Medallion.

**A decisão de modelagem mais importante não foi técnica.**  
O mix de modalidades tem distribuições de valor incomparáveis:  
- Fiscal: importância segurada de R$ 500k a R$ 50M  
- Recursal: importância média de R$ 3.000  

Agregar as duas num único ticket médio produz um número que não representa nenhum dos dois segmentos. É estatisticamente correto e estrategicamente inútil.

---

## Decisão de Arquitetura

| Modelo | Considerado | Decisão |
|--------|-------------|--------|
| **Star Schema** | ✅ | **Escolhido** — leitura analítica otimizada, linguagem familiar para BI e stakeholders |
| Data Vault | ✅ | Descartado — over-engineering para volume e perfil de consumo analítico |
| OBT (One Big Table) | ✅ | Descartado — impossível versionar dimensões sem reescrever a fato |

---

## Modelo

```
              dim_tempo
                  │
dim_cliente ── FATO_FUNIL ── dim_produto
              ORIGINACAO
                  │
              dim_canal

Atributos degenerados na fato:
  importancia_segurada, premio_calculado,
  taxa_comissao*, valor_comissao_reais

* taxa_comissao é degenerada porque varia por
  seguradora × cliente — não descreve o canal
  nem o produto, descreve a negociação específica.
```

## 0. Configuração

In [1]:
import sys
sys.path.append("../src")

import findspark
findspark.init()

import random
import uuid
from datetime import date, datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils import SEED, SEGURADORAS, MIX_MODALIDADE, gerar_id_funil, data_aleatoria

random.seed(SEED)

spark = (
    SparkSession.builder
    .appName("DataFoundation_03_StarSchema")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version}")

✅ Spark 4.1.1


## 1. Geração dos Dados — Funil de Originação

Parâmetros baseados em operação real:
- 100.000 cotações/mês | Mix 10/70/20
- Cotação → Proposta: 45% | Proposta → Emissão: 64% (~29% do topo)

In [2]:
N_COTACOES    = 100_000
TAXA_PROPOSTA = 0.45
TAXA_EMISSAO  = 0.644

IS_RANGES = {
    "fiscal"  : (500_000, 50_000_000),
    "recursal": (1_000,   10_000),
    "outras"  : (50_000,  2_000_000),
}
TAXA_PREMIO = {
    "fiscal"  : (0.0008, 0.0025),
    "recursal": (0.0100, 0.0300),
    "outras"  : (0.0015, 0.0050),
}
CLIENTES = [f"CLI{str(i).zfill(5)}" for i in range(1, 5_001)]
PORTES   = ["micro", "pequeno", "medio", "grande", "corporativo"]
SETORES  = ["construcao_civil", "servicos", "industria", "comercio", "energia"]

registros = []
for i in range(1, N_COTACOES + 1):
    modalidade = random.choices(list(MIX_MODALIDADE.keys()), weights=list(MIX_MODALIDADE.values()))[0]
    seguradora = random.choice(SEGURADORAS)
    id_cliente = random.choice(CLIENTES)
    data_cot   = data_aleatoria("2023-01-01", "2024-12-31")

    is_min, is_max = IS_RANGES[modalidade]
    importancia    = round(random.uniform(is_min, is_max), 2)
    tp_min, tp_max = TAXA_PREMIO[modalidade]
    premio         = round(importancia * random.uniform(tp_min, tp_max), 2)
    taxa_com       = round(random.uniform(seguradora["comissao_min"], seguradora["comissao_max"]), 4)
    valor_com      = round(premio * taxa_com, 2)
    id_funil       = gerar_id_funil()

    base = {"id_funil": id_funil, "id_cliente": id_cliente, "modalidade": modalidade,
            "seguradora": seguradora["nome"], "importancia_segurada": importancia,
            "premio_calculado": premio, "taxa_comissao": taxa_com}

    registros.append({**base, "estagio": "cotacao", "valor_comissao_reais": 0.0,
                      "flag_converteu": False, "data_evento": data_cot})

    if random.random() < TAXA_PROPOSTA:
        data_prop = data_aleatoria(data_cot, "2024-12-31")
        registros.append({**base, "estagio": "proposta", "valor_comissao_reais": 0.0,
                          "flag_converteu": False, "data_evento": data_prop})

        if random.random() < TAXA_EMISSAO:
            data_em = data_aleatoria(data_prop, "2024-12-31")
            registros.append({**base, "estagio": "emissao", "valor_comissao_reais": valor_com,
                              "flag_converteu": True, "data_evento": data_em})

df_staging = spark.createDataFrame(registros)
n_cot  = df_staging.filter(F.col("estagio") == "cotacao").count()
n_prop = df_staging.filter(F.col("estagio") == "proposta").count()
n_em   = df_staging.filter(F.col("estagio") == "emissao").count()

print(f"✅ Staging: {df_staging.count():,} registros")
print(f"\n  Cotações  : {n_cot:,}  (100%)")
print(f"  Propostas : {n_prop:,}  ({n_prop/n_cot*100:.1f}%)")
print(f"  Emissões  : {n_em:,}   ({n_em/n_cot*100:.1f}%)")

✅ Staging: 174,561 registros

  Cotações  : 100,000  (100%)
  Propostas : 45,236  (45.2%)
  Emissões  : 29,325   (29.3%)


## 2. Dimensões

In [3]:
# dim_produto
dim_produto = spark.createDataFrame([
    {"sk_produto": 1, "modalidade": "fiscal",   "perfil_is": "alto_valor",  "prazo_medio_vigencia": 36},
    {"sk_produto": 2, "modalidade": "recursal", "perfil_is": "baixo_valor", "prazo_medio_vigencia": 18},
    {"sk_produto": 3, "modalidade": "outras",   "perfil_is": "medio_valor", "prazo_medio_vigencia": 24},
])

# dim_canal
dim_canal = spark.createDataFrame([
    {"sk_canal": 1, "seguradora": "Tokio Marine",  "rating": "AAA"},
    {"sk_canal": 2, "seguradora": "Porto Seguro",  "rating": "AA+"},
    {"sk_canal": 3, "seguradora": "Swiss Re",      "rating": "AAA"},
    {"sk_canal": 4, "seguradora": "Junto Seguros", "rating": "A+"},
    {"sk_canal": 5, "seguradora": "SulAmérica",    "rating": "AA"},
])

# dim_cliente — seed fixo para reprodutibilidade (porte/setor não mudam entre execuções)
random.seed(SEED)
dim_cliente = spark.createDataFrame([
    {"sk_cliente": i+1, "id_cliente": f"CLI{str(i+1).zfill(5)}",
     "porte": random.choice(PORTES), "setor": random.choice(SETORES),
     "segmento": random.choice(["varejo", "corporate", "middle"])}
    for i in range(5_000)
])

# dim_tempo
from datetime import date, timedelta
datas, d, sk = [], date(2023, 1, 1), 1
while d <= date(2024, 12, 31):
    datas.append({"sk_tempo": sk, "data": d.strftime("%Y-%m-%d"),
                  "ano": d.year, "trimestre": (d.month-1)//3+1,
                  "mes": d.month, "flag_util": d.weekday() < 5})
    d += timedelta(days=1); sk += 1
dim_tempo = spark.createDataFrame(datas)

print(f"✅ dim_produto  : {dim_produto.count()} linhas")
print(f"✅ dim_canal    : {dim_canal.count()} linhas")
print(f"✅ dim_cliente  : {dim_cliente.count():,} linhas")
print(f"✅ dim_tempo    : {dim_tempo.count():,} linhas")

✅ dim_produto  : 3 linhas
✅ dim_canal    : 5 linhas
✅ dim_cliente  : 5,000 linhas
✅ dim_tempo    : 731 linhas


## 3. Tabela Fato

In [4]:
fato = (
    df_staging
    .join(dim_produto.select("sk_produto", "modalidade"), on="modalidade", how="left")
    .join(dim_canal.select("sk_canal", "seguradora"),     on="seguradora", how="left")
    .join(dim_cliente.select("sk_cliente", "id_cliente"), on="id_cliente", how="left")
    .join(dim_tempo.select("sk_tempo", "data"),
          df_staging["data_evento"] == dim_tempo["data"], how="left")
    .select(
        "id_funil", "sk_tempo", "sk_produto", "sk_canal", "sk_cliente",
        "estagio", "importancia_segurada", "premio_calculado",
        "taxa_comissao", "valor_comissao_reais", "flag_converteu", "data_evento"
    )
)

fato.cache()
print(f"✅ Fato construída: {fato.count():,} registros")
fato.printSchema()

✅ Fato construída: 174,561 registros
root
 |-- id_funil: string (nullable = true)
 |-- sk_tempo: long (nullable = true)
 |-- sk_produto: long (nullable = true)
 |-- sk_canal: long (nullable = true)
 |-- sk_cliente: long (nullable = true)
 |-- estagio: string (nullable = true)
 |-- importancia_segurada: double (nullable = true)
 |-- premio_calculado: double (nullable = true)
 |-- taxa_comissao: double (nullable = true)
 |-- valor_comissao_reais: double (nullable = true)
 |-- flag_converteu: boolean (nullable = true)
 |-- data_evento: string (nullable = true)



## 4. Governança — Ownership e Classificação

**Ownership do domínio Funil de Originação:**

| Domínio | Data Owner | Data Steward | SLA de qualidade |
|---|---|---|---|
| Funil de originação | Diretoria Comercial | Especialista em Dados | Conversão monitorada semanalmente |
| Cadastro de segurados | Compliance / DPO | Analista de Dados | Completude ≥ 98% |
| Comissões por seguradora | Diretoria Financeira | Especialista em Dados | Reconciliação mensal obrigatória |

**Classificação dos campos da Fato:**

| Campo | Classificação | Controle em produção |
|---|---|---|
| `valor_comissao_reais` | **Confidencial** — dado financeiro sensível | RBAC, log de acesso, criptografia em repouso |
| `taxa_comissao` | **Confidencial** — dado comercial estratégico | Acesso restrito a Financeiro e Diretoria |
| `id_cliente` | **Confidencial** — chave de negócio | RBAC + pseudonimização na exposição |
| `importancia_segurada` | **Interno** | Autenticação básica |
| `estagio`, `modalidade` | **Interno** | Autenticação básica |
| `sk_*` (surrogate keys) | **Interno** | Autenticação básica |

**Data Dictionary — campos críticos da Fato:**

| Campo | Definição de negócio | Regra de validação |
|---|---|---|
| `estagio` | Posição no funil de originação | Enum: cotacao → proposta → emissão. Sem retrospecção. |
| `taxa_comissao` | Percentual de comissão negociado por seguradora × cliente | Entre comissao_min e comissao_max da seguradora. Degenerado na fato. |
| `valor_comissao_reais` | Receita de comissão gerada na emissão | Zero nos estágios cotacao e proposta. Positivo apenas em emissao. |
| `flag_converteu` | Indica se a operação gerou receita | True apenas em emissao com valor_comissao_reais > 0 |


## 4. Persistência — Camada Gold

In [5]:
import os
from datetime import datetime

os.makedirs("../data/gold", exist_ok=True)

_dt_proc = datetime.now().strftime("%Y-%m-%d %H:%M")

# Caminho B: toPandas() para contornar limitação do winutils no Windows
# Em produção Linux/cloud: .write.format("delta").partitionBy("sk_tempo").save(...)
tabelas = {
    "fato_funil_originacao": fato,
    "dim_produto":  dim_produto,
    "dim_canal":    dim_canal,
    "dim_tempo":    dim_tempo,
    "dim_cliente":  dim_cliente,
}

for nome, df in tabelas.items():
    df_pd = df.toPandas()
    # Colunas de linhagem em todas as tabelas Gold
    df_pd["_origem"]           = f"NB03_modelagem_dimensional"
    df_pd["_dt_processamento"] = _dt_proc
    df_pd.to_csv(f"../data/gold/{nome}.csv", index=False)
    print(f"  ✅ {nome} — {len(df_pd):,} linhas")

print(f"\n  _origem           : NB03_modelagem_dimensional")
print(f"  _dt_processamento : {_dt_proc}")
print("\n💡 Em produção: .write.format('delta').partitionBy('sk_tempo')")

spark.stop()
print("\n✅ SparkSession encerrada. Próximo: 04_funil_originacao.ipynb")


  ✅ fato_funil_originacao — 174,561 linhas
  ✅ dim_produto — 3 linhas
  ✅ dim_canal — 5 linhas
  ✅ dim_tempo — 731 linhas
  ✅ dim_cliente — 5,000 linhas

  _origem           : NB03_modelagem_dimensional
  _dt_processamento : 2026-06-30 15:39

💡 Em produção: .write.format('delta').partitionBy('sk_tempo')

✅ SparkSession encerrada. Próximo: 04_funil_originacao.ipynb
